# IMDB Dataset

In [ ]:
vocab_size=10000

from tensorflow.keras.datasets import imdb

(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=vocab_size)

17464789/17464789 [==============================] - 0s 0us/step


In [ ]:
word_index = imdb.get_word_index()
reverse_word_index = dict(
    [(value, key) for (key, value) in word_index.items()])

1641221/1641221 [==============================] - 0s 0us/step


In [ ]:
for doc_index in range(0,10):
  decoded_review = " ".join([reverse_word_index.get(i - 3, "?") for i in train_data[doc_index]])
  print(f'[{doc_index}]: {decoded_review}')

[0]: ? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't yo

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Flatten, Dense

vocab_size = 10000
max_length = 50
embedding_dim = 8

train_data = pad_sequences(train_data, maxlen=max_length)
test_data = pad_sequences(test_data, maxlen=max_length)

model = Sequential()
model.add(Embedding(vocab_size, embedding_dim, input_length=max_length))
model.add(SimpleRNN(16, return_sequences=True))
model.add(SimpleRNN(32))
model.add(Flatten())
model.add(Dense(1, activation='sigmoid'))

print("Model Architecture:")
print(model.summary())

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.fit(train_data, train_labels,
          epochs=5,
          batch_size=64,
          validation_data=(test_data, test_labels))

y_pred_prob = model.predict(test_data)
y_pred = np.round(y_pred_prob).astype(int)

cm = confusion_matrix(test_labels, y_pred)
print("Confusion Matrix:")
print(cm)

report = classification_report(test_labels, y_pred)
print("Classification Report:")
print(report)

Model Architecture:
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 50, 8)             80000     
                                                                 
 simple_rnn_2 (SimpleRNN)    (None, 50, 16)            400       
                                                                 
 simple_rnn_3 (SimpleRNN)    (None, 32)                1568      
                                                                 
 flatten_1 (Flatten)         (None, 32)                0         
                                                                 
 dense_1 (Dense)             (None, 1)                 33        
                                                                 
Total params: 82001 (320.32 KB)
Trainable params: 82001 (320.32 KB)
Non-trainable params: 0 (0.00 Byte)
______________________________________________________________

#Persian Dataset

In [1]:
!pip install datasets
from datasets import load_dataset

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 13.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.20.3
    Uninstalling huggingface-hub-0.20.3:
      Successfully uninstalled huggingface-hub-0.20.3


In [5]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Flatten, Dense
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from datasets import load_dataset

persian_dataset = load_dataset("SeyedAli/Persian-Text-Sentiment")

train_texts = [example['text'] for example in persian_dataset['train']]
train_labels = [example['label'] for example in persian_dataset['train']]
test_texts = [example['text'] for example in persian_dataset['test']]
test_labels = [example['label'] for example in persian_dataset['test']]

vocab_size = 10000
max_length = 50
embedding_dim = 8

train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(train_texts)

train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

train_data = pad_sequences(train_sequences, maxlen=max_length, padding='post', truncating='post')
test_data = pad_sequences(test_sequences, maxlen=max_length, padding='post', truncating='post')

def evaluate_model(model, test_data, test_labels):
    predictions = model.predict(test_data)
    predictions_binary = [1 if p > 0.5 else 0 for p in predictions]

    accuracy = accuracy_score(test_labels, predictions_binary)
    precision = precision_score(test_labels, predictions_binary)
    recall = recall_score(test_labels, predictions_binary)
    f1 = f1_score(test_labels, predictions_binary)

    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)

    cm = confusion_matrix(test_labels, predictions_binary)
    print("Confusion Matrix:")
    print(cm)

    return accuracy

model = Sequential()
model.add(Embedding(vocab_size, embedding_dim, input_length=max_length))
model.add(SimpleRNN(16, return_sequences=True))
model.add(SimpleRNN(32))
model.add(Flatten())
model.add(Dense(1, activation='sigmoid'))

print(f"Embedding Dimension: {embedding_dim}")
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.fit(train_data, train_labels,
          epochs=5,
          batch_size=64,
          validation_data=(test_data, test_labels))

accuracy = evaluate_model(model, test_data, test_labels)
print(f"Final Accuracy for Embedding Dimension {embedding_dim}: {accuracy}\n")

Embedding Dimension: 8
Epoch 1/5
873/873 [==============================] - 164s 183ms/step - loss: 0.4712 - accuracy: 0.7651 - val_loss: 0.3562 - val_accuracy: 0.8498
Epoch 2/5
873/873 [==============================] - 88s 101ms/step - loss: 0.3325 - accuracy: 0.8646 - val_loss: 0.3474 - val_accuracy: 0.8553
Epoch 3/5
873/873 [==============================] - 84s 97ms/step - loss: 0.3021 - accuracy: 0.8790 - val_loss: 0.3553 - val_accuracy: 0.8522
Epoch 4/5
873/873 [==============================] - 84s 96ms/step - loss: 0.2853 - accuracy: 0.8861 - val_loss: 0.3869 - val_accuracy: 0.8485
Epoch 5/5
437/437 [==============================] - 5s 10ms/step
Accuracy: 0.8393010598682326
Precision: 0.8442313281022958
Recall: 0.8321397880263535
F1 Score: 0.8381419503750721
Confusion Matrix:
[[5910 1072]
 [1172 5810]]
Final Accuracy for Embedding Dimension 8: 0.8393010598682326



#Third Dataset

In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense,SimpleRNN
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import numpy as np

dataset = load_dataset("saied/persian_news_dataset", split="train", streaming=True)

docs = []
document_count = 1000
counter = 0
skip_counter = 0

for doc in dataset:
    if len(doc['title']) == 0 or len(doc['category']) == 0:
        skip_counter += 1
        if skip_counter % 10000 == 0:
            print(f'{skip_counter} skipped')
        continue
    docs.append(doc)
    counter += 1
    if counter == document_count:
        break

texts = [doc['text'] for doc in docs]
categories = [doc['category'] for doc in docs]

max_words = 10000
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

input_length = 50
X = pad_sequences(sequences, maxlen=input_length)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(categories)
number_of_classes = len(label_encoder.classes_)
y = tf.keras.utils.to_categorical(y, num_classes=number_of_classes)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

embedding_dim = 100

model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=input_length))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(number_of_classes, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_val, y_val), verbose=2)

val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)
y_pred = np.argmax(model.predict(X_val), axis=1)
y_true = np.argmax(y_val, axis=1)

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')
cm = confusion_matrix(y_true, y_pred)

print(f"Accuracy: {val_accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")
print(f"Confusion Matrix:\n{cm}")
print(f"Final Accuracy for Embedding Dimension {embedding_dim}: {val_accuracy}")

/usr/local/lib/python3.10/dist-packages/datasets/load.py:1486: FutureWarning: The repository for saied/persian_news_dataset contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/saied/persian_news_dataset
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


10000 skipped
20000 skipped
30000 skipped
40000 skipped
50000 skipped
60000 skipped
70000 skipped
80000 skipped
90000 skipped
100000 skipped
110000 skipped
120000 skipped
130000 skipped
140000 skipped
150000 skipped
160000 skipped
170000 skipped
180000 skipped
190000 skipped
200000 skipped
210000 skipped
Epoch 1/5
25/25 - 6s - loss: 1.5796 - accuracy: 0.4650 - val_loss: 1.3174 - val_accuracy: 0.6400 - 6s/epoch - 227ms/step
Epoch 2/5
25/25 - 2s - loss: 0.6339 - accuracy: 0.9100 - val_loss: 0.5304 - val_accuracy: 0.8600 - 2s/epoch - 96ms/step
Epoch 3/5
25/25 - 2s - loss: 0.0818 - accuracy: 1.0000 - val_loss: 0.2327 - val_accuracy: 0.9400 - 2s/epoch - 78ms/step
Epoch 4/5
25/25 - 1s - loss: 0.0158 - accuracy: 1.0000 - val_loss: 0.1794 - val_accuracy: 0.9500 - 1s/epoch - 59ms/step
Epoch 5/5
25/25 - 1s - loss: 0.0076 - accuracy: 1.0000 - val_loss: 0.1514 - val_accuracy: 0.9650 - 558ms/epoch - 22ms/step
7/7 [==============================] - 0s 2ms/step
Accuracy: 0.9649999737739563
Precision: